Compare conformity measures in the original and full clinical protocols.....


In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import io
import json
import os
import sys
import warnings
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings('ignore')
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/panel_mechanisms/criterion_recovery_and_difficulty'))
os.makedirs(OUT, exist_ok=True)
SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_noroles_%s_judge.csv'
PBASE = {'non': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated/llm_as_judge/delphi_base_noroles_judge.csv', 'full': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv'}
HOMOG_DS = {'non': 'experiments/homogeneous_panels/three_deepseekv4_members_truncated/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv', 'full': 'experiments/homogeneous_panels/three_deepseekv4_members_complete/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'}
FIXED = {'P-weak': ('experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/delphi_pweak_noroles_judge.csv', ['qwen', 'gemini', 'deepseekv4pro']), 'P-mixed': ('experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/delphi_pmixed_noroles_judge.csv', ['deepseekv4', 'gemini', 'glm']), 'P-strong': ('experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/delphi_pstrong_noroles_judge.csv', ['deepseekv4', 'glm', 'gpt']), 'homog gpt': ('experiments/homogeneous_panels/three_gpt_members/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', ['gpt'])}

def rd(rel):
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        return None
    d = read_csv_normalized(p, sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    if 'judge_details' not in d.columns:
        return None
    return d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette').set_index('Vignette')

def items(d):
    r = {}
    for v, s in d['judge_details'].dropna().items():
        try:
            arr = json.loads(s)
        except Exception:
            continue
        for it in arr:
            r[v, it['id']] = (bool(it.get('met')), it.get('points', 0) or 0)
    return r
SING = {m: items(rd(NB % (m, m))) for m in SINGLES}

def cmh(tabs):
    tabs = [t for t in tabs if t.sum() > 0]
    num = sum((t[0, 0] * t[1, 1] / t.sum() for t in tabs))
    den = sum((t[0, 1] * t[1, 0] / t.sum() for t in tabs))
    if den == 0 or num == 0:
        return np.nan
    return num / den

def build(panel_items, members, keys):
    rows = []
    held = [m for m in SINGLES if m not in members]
    for k in keys:
        rows.append({'diff': 1 - np.mean([SING[m][k][0] for m in held]), 'panel': panel_items[k][0], 'mem_mean': np.mean([SING[m][k][0] for m in members]), 'best_mem': max((SING[m][k][0] for m in members)), 'points': panel_items[k][1]})
    d = pd.DataFrame(rows)
    d['strat'] = pd.cut(d['diff'], [-0.01, 0.001, 0.26, 0.51, 0.76, 1.01], labels=['очень лёгкие', 'лёгкие', 'средние', 'трудные', 'очень трудные'])
    return d

def report(d, title):
    print('\n' + '-' * 100)
    print(title + '   (пунктов: %d)' % len(d))
    print('-' * 100)
    t = d.groupby('strat', observed=True).agg(пунктов=('panel', 'size'), панель=('panel', 'mean'), участники=('mem_mean', 'mean'), лучший=('best_mem', 'mean'))
    t['панель−участники'] = t['панель'] - t['участники']
    t['панель−лучший'] = t['панель'] - t['лучший']
    print(t.round(4).to_string())
    res = []
    for lab, s in d.groupby('strat', observed=True):
        tabs = [np.array([[int(r.panel), 1 - int(r.panel)], [int(r.best_mem), 1 - int(r.best_mem)]]) for _, r in s.iterrows()]
        b = int((s.panel & ~s.best_mem.astype(bool)).sum())
        c = int((~s.panel.astype(bool) & s.best_mem.astype(bool)).sum())
        res.append({'страта': lab, 'пунктов': len(s), 'OR (МХ)': cmh(tabs), 'панель+ / лучший−': b, 'панель− / лучший+': c, 'p (McNemar)': stats.binomtest(b, b + c, 0.5).pvalue if b + c else np.nan})
    r = pd.DataFrame(res).set_index('страта')
    print('\n' + r.to_string())
    disc = d[d.panel != d.best_mem]
    ct = pd.crosstab(disc.strat, disc.panel)
    chi = stats.chi2_contingency(ct)
    print('\n  доля «панель выиграла» среди расходящихся: ' + ' | '.join(('%s %.3f' % (i, v) for i, v in (ct[True] / ct.sum(axis=1)).items())))
    print('  однородность: chi2 = %.2f, df = %d, p = %.3g' % (chi.statistic, chi.dof, chi.pvalue))
    long = pd.concat([pd.DataFrame({'met': d.panel.astype(int), 'arm': 'panel', 'diff': d['diff'], 'points': d.points}), pd.DataFrame({'met': d.best_mem.astype(int), 'arm': 'best_mem', 'diff': d['diff'], 'points': d.points})])
    m = smf.logit("met ~ C(arm, Treatment('best_mem')) * diff + points", long).fit(disp=0)
    tab = pd.DataFrame({'коэф': m.params, 'SE': m.bse, 'z': m.tvalues, 'p': m.pvalues, 'OR': np.exp(m.params)})
    print('\n' + tab.round(4).to_string())
    print('  n = %d, псевдо-R2 = %.3f' % (int(m.nobs), m.prsquared))
    ik = [i for i in m.params.index if ':' in i][0]
    return {'взаимодействие': m.params[ik], 'p': m.pvalues[ik], 'OR взаим.': np.exp(m.params[ik]), 'chi2 однородности': chi.statistic, 'p однородности': chi.pvalue, 'OR лёгкие': r.loc['очень лёгкие', 'OR (МХ)'], 'OR оч.трудные': r.loc['очень трудные', 'OR (МХ)']}
print('=' * 100)
print('H3.7 В ДВУХ ВАРИАНТАХ: ГРАДИЕНТ ПРИРОСТА ПАНЕЛИ ПО ТРУДНОСТИ ПУНКТА')
print('=' * 100)
summary = []
for v in ['non', 'full']:
    P = items(rd(PBASE[v]))
    H = items(rd(HOMOG_DS[v]))
    keysP = sorted([k for k in P if P[k][1] > 0 and all((k in SING[m] for m in SINGLES))])
    keysH = sorted([k for k in H if H[k][1] > 0 and all((k in SING[m] for m in SINGLES))])
    s1 = report(build(P, ['gpt', 'qwen', 'deepseekv4'], keysP), 'P-base, вариант %s' % v.upper())
    s1.update({'плечо': 'P-base', 'вариант': v})
    s2 = report(build(H, ['deepseekv4'], keysH), 'homog deepseekv4, вариант %s' % v.upper())
    s2.update({'плечо': 'homog ds-v4', 'вариант': v})
    summary += [s1, s2]
print('\n' + '=' * 100)
print('НЕИЗМЕНЁННЫЕ ПЛЕЧИ (одинаковы в обоих вариантах) -- контроль')
print('=' * 100)
for name, (path, mem) in FIXED.items():
    d = rd(path)
    if d is None:
        continue
    it = items(d)
    keys = sorted([k for k in it if it[k][1] > 0 and all((k in SING[m] for m in SINGLES))])
    s = report(build(it, mem, keys), name)
    s.update({'плечо': name, 'вариант': '—'})
    summary.append(s)
print('\n' + '=' * 100)
print('СВОДКА')
print('=' * 100)
S = pd.DataFrame(summary)[['плечо', 'вариант', 'OR лёгкие', 'OR оч.трудные', 'взаимодействие', 'p', 'chi2 однородности', 'p однородности']]
print(S.round(4).to_string(index=False))
S.to_csv(os.path.normpath(os.path.join(OUT, 'truncated_vs_complete_difficulty_effects.csv')), index=False, encoding='utf-8-sig')
print('\nCSV: %s' % os.path.normpath(os.path.join(OUT, 'truncated_vs_complete_difficulty_effects.csv')))


H3.7 В ДВУХ ВАРИАНТАХ: ГРАДИЕНТ ПРИРОСТА ПАНЕЛИ ПО ТРУДНОСТИ ПУНКТА



----------------------------------------------------------------------------------------------------
P-base, вариант NON   (пунктов: 2249)
----------------------------------------------------------------------------------------------------
               пунктов  панель  участники  лучший  панель−участники  панель−лучший
strat                                                                             
очень лёгкие       727  0.9835     0.9629  0.9986            0.0206        -0.0151
лёгкие             267  0.9363     0.8227  0.9850            0.1136        -0.0487
средние            234  0.7863     0.6182  0.8974            0.1681        -0.1111
трудные            339  0.6932     0.3697  0.6696            0.3235         0.0236
очень трудные      682  0.4311     0.1281  0.2933            0.3030         0.1378



               пунктов   OR (МХ)  панель+ / лучший−  панель− / лучший+   p (McNemar)
страта                                                                              
очень лёгкие       727  0.083333                  1                 12  3.417969e-03
лёгкие             267  0.235294                  4                 17  7.197380e-03
средние            234  0.365854                 15                 41  6.855642e-04
трудные            339  1.177778                 53                 45  4.796925e-01
очень трудные      682  2.287671                167                 73  1.198460e-09

  доля «панель выиграла» среди расходящихся: очень лёгкие 0.077 | лёгкие 0.190 | средние 0.268 | трудные 0.541 | очень трудные 0.696
  однородность: chi2 = 61.49, df = 4, p = 1.41e-12

                                               коэф      SE        z    p       OR
Intercept                                    4.5612  0.3416  13.3504  0.0  95.6943
C(arm, Treatment('best_mem'))[T.panel]      -1.9147 


               пунктов   OR (МХ)  панель+ / лучший−  панель− / лучший+   p (McNemar)
страта                                                                              
очень лёгкие       672  1.285714                  9                  7  8.036194e-01
лёгкие             218  1.857143                 13                  7  2.631760e-01
средние            343  1.341463                 55                 41  1.842859e-01
трудные            195  2.160000                 54                 25  1.466072e-03
очень трудные      821  2.824561                161                 57  1.121281e-12

  доля «панель выиграла» среди расходящихся: очень лёгкие 0.562 | лёгкие 0.650 | средние 0.573 | трудные 0.684 | очень трудные 0.739
  однородность: chi2 = 9.60, df = 4, p = 0.0477

                                               коэф      SE        z       p       OR
Intercept                                    2.8645  0.2481  11.5471  0.0000  17.5398
C(arm, Treatment('best_mem'))[T.panel]      -0.01


----------------------------------------------------------------------------------------------------
P-base, вариант FULL   (пунктов: 2249)
----------------------------------------------------------------------------------------------------
               пунктов  панель  участники  лучший  панель−участники  панель−лучший
strat                                                                             
очень лёгкие       727  0.9876     0.9629  0.9986            0.0248        -0.0110
лёгкие             267  0.9551     0.8227  0.9850            0.1323        -0.0300
средние            234  0.8205     0.6182  0.8974            0.2023        -0.0769
трудные            339  0.6814     0.3697  0.6696            0.3117         0.0118
очень трудные      682  0.4355     0.1281  0.2933            0.3074         0.1422


...


               пунктов   OR (МХ)  панель+ / лучший−  панель− / лучший+   p (McNemar)
страта                                                                              
очень лёгкие       816  0.142857                  1                  7  7.031250e-02
лёгкие             271  1.388889                 25                 18  3.603777e-01
средние            237  1.458333                 35                 24  1.925265e-01
трудные            339  1.822222                 82                 45  1.307385e-03
очень трудные      586  3.818182                168                 44  2.886145e-18

  доля «панель выиграла» среди расходящихся: очень лёгкие 0.125 | лёгкие 0.581 | средние 0.593 | трудные 0.646 | очень трудные 0.792
  однородность: chi2 = 28.58, df = 4, p = 9.49e-06

                                               коэф      SE        z       p       OR
Intercept                                    3.1208  0.2658  11.7404  0.0000  22.6637
C(arm, Treatment('best_mem'))[T.panel]      -0


----------------------------------------------------------------------------------------------------
P-strong   (пунктов: 2249)
----------------------------------------------------------------------------------------------------
               пунктов  панель  участники  лучший  панель−участники  панель−лучший
strat                                                                             
очень лёгкие       710  0.9901     0.9723  0.9986            0.0178        -0.0085
лёгкие             274  0.9635     0.8273  0.9745            0.1363        -0.0109
средние            243  0.8683     0.6433  0.9053            0.2250        -0.0370
трудные            324  0.7253     0.3704  0.6821            0.3549         0.0432
очень трудные      698  0.4570     0.1409  0.3095            0.3161         0.1476



               пунктов   OR (МХ)  панель+ / лучший−  панель− / лучший+   p (McNemar)
страта                                                                              
очень лёгкие       710       NaN                  0                  6  3.125000e-02
лёгкие             274  0.666667                  6                  9  6.072388e-01
средние            243  0.625000                 15                 24  1.995909e-01
трудные            324  1.291667                 62                 48  2.149765e-01
очень трудные      698  2.303797                182                 79  1.588017e-10

  доля «панель выиграла» среди расходящихся: очень лёгкие 0.000 | лёгкие 0.400 | средние 0.385 | трудные 0.564 | очень трудные 0.697
  однородность: chi2 = 29.95, df = 4, p = 5.02e-06

                                               коэф      SE        z      p       OR
Intercept                                    3.9606  0.3325  11.9112  0.000  52.4913
C(arm, Treatment('best_mem'))[T.panel]      -1.1


----------------------------------------------------------------------------------------------------
homog gpt   (пунктов: 2249)
----------------------------------------------------------------------------------------------------
               пунктов  панель  участники  лучший  панель−участники  панель−лучший
strat                                                                             
очень лёгкие       677  0.9808     0.9734  0.9734            0.0074         0.0074
лёгкие             228  0.9123     0.8465  0.8465            0.0658         0.0658
средние            329  0.8116     0.6687  0.6687            0.1429         0.1429
трудные            195  0.6974     0.4718  0.4718            0.2256         0.2256
очень трудные      820  0.4427     0.1841  0.1841            0.2585         0.2585



               пунктов   OR (МХ)  панель+ / лучший−  панель− / лучший+   p (McNemar)
страта                                                                              
очень лёгкие       677  1.625000                 13                  8  3.833103e-01
лёгкие             228  2.666667                 24                  9  1.353099e-02
средние            329  2.880000                 72                 25  1.918739e-06
трудные            195  3.750000                 60                 16  3.850978e-07
очень трудные      820  5.510638                259                 47  1.174148e-36

  доля «панель выиграла» среди расходящихся: очень лёгкие 0.619 | лёгкие 0.727 | средние 0.742 | трудные 0.789 | очень трудные 0.846
  однородность: chi2 = 11.68, df = 4, p = 0.0199

                                               коэф      SE        z       p      OR
Intercept                                    2.2074  0.2195  10.0576  0.0000  9.0924
C(arm, Treatment('best_mem'))[T.panel]       0.302